# Querying USPTO for Plant Patent Metadata and Color Images

Author: Alexander Maksiaev \
Purpose: Use USPTO API to query database to find plant patent metadata and color images (note: API only has data from 2001 onward)

In [ ]:
import os
import json
import requests
import time
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env

API_KEY = os.environ.get("API_KEY")
HTTP_RETRY = 5 # Number of times to retry a request
SLEEP_AFTER_429 = 30 # Seconds to sleep if too many requests

url = 'https://api.uspto.gov/api/v1'
application_id = "17697900" # Application ID finds the data -- now need to get from Patent Number to Application ID
meta_data = f"{url}/patent/applications/{application_id}/documents"

headers = {
    'x-api-key': API_KEY # Needed for permission
}

files  = [meta_data] # This should be a list of metadata links (length equal to number of patents) -- but for now it's just the example

# Function to make a search request
def make_search_request(file_url, retry=0):
    global total_429 
    response = requests.get(file_url, headers=headers, allow_redirects=True) # Allow redirects allows us to download the file
    if response.status_code == 429 and retry < HTTP_RETRY: # HTTP 429 = "Too many requests"
        time.sleep(SLEEP_AFTER_429)
        retry += 1
        total_429 += 1
        print(f"Got HTTP 429. Retry number: {retry}")
        return make_search_request(file_url, retry)
    return response

# Loop over application numbers (ideally would be patent numbers)
for count in range(0, len(files)): 
    file_url = files[count]
    response = make_search_request(file_url, 0)
    # time.sleep(SLEEP_BETWEEN_HTTP)
    if response.status_code == 200:
        print(f"Retrieved {count}: {file_url}, {response.content}")
        try:
            download_url = json.loads(response.content)["documentBag"][-1]["downloadOptionBag"][-1]["downloadUrl"] # Find color image, if exists
            print(download_url) 
            download = make_search_request(download_url, 0)
            if download.status_code == 200:
                print("success")
                with open(application_id + ".pdf", "wb") as out:
                    out.write(download.content) # Download the image as a pdf
            else:
                print("Failed to reach color image")
                continue
        except:
            print("Failed to find color image")
            continue
    else:
        print(f"request failed with status code {response.status_code} for url: {file_url}, {response.content}")



Retrieved 0: https://api.uspto.gov/api/v1/patent/applications/17697900/documents, b'{"count": 56, "documentBag": [{"applicationNumberText": "17697900", "officialDate": "2022-12-14T14:38:09.000-0500", "documentIdentifier": "LBO21YLVXBLUEX3", "documentCode": "ISSUE.NTF", "documentCodeDescriptionText": "Issue Notification", "directionCategory": "OUTGOING", "downloadOptionBag": [{"mimeTypeIdentifier": "PDF", "downloadUrl": "https://api.uspto.gov/api/v1/download/applications/17697900/LBO21YLVXBLUEX3.pdf", "pageTotalQuantity": 1}]}, {"applicationNumberText": "17697900", "officialDate": "2022-11-01T12:04:04.000-0400", "documentIdentifier": "L9YEGZU9XBLUEX2", "documentCode": "IFEE", "documentCodeDescriptionText": "Issue Fee Payment (PTO-85B)", "directionCategory": "INCOMING", "downloadOptionBag": [{"mimeTypeIdentifier": "PDF", "downloadUrl": "https://api.uspto.gov/api/v1/download/applications/17697900/L9YEGZU9XBLUEX2.pdf", "pageTotalQuantity": 1}]}, {"applicationNumberText": "17697900", "offic